# 04 · NLP profiles and hybrid recommender — demo + evaluation
Uses pipeline artifacts (`python -m ml.rebuild_all`). Evaluation: leave-one-out on ratings — hide one highly-rated destination per user, then check whether a content-only probe built from their *other* likes ranks it in the top-K.

In [1]:
import os, sys, json, warnings
warnings.filterwarnings("ignore")
ROOT = r"C:\xampp\htdocs\travel_journel"
os.chdir(ROOT); sys.path.insert(0, ROOT)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 70)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": .3})
DATA = os.path.join(ROOT, "ml", "data")

In [2]:
import pickle
prof = pickle.load(open("ml/artifacts/dest_profiles.pkl","rb"))["profiles"]
P = pd.DataFrame([{k:v for k,v in p.items() if k in ("canonical_dest","n_journals","region","popularity_tier","avg_daily","sentiment_mean")} for p in prof.values()])
print(len(P), "destination profiles")
P.region.value_counts()

561 destination profiles


region
West India         151
South India        141
East India          75
North India         67
Himalayas           59
Northeast India     25
Central India       16
Southeast Asia      12
Europe               8
South Asia           3
Caucasus             2
Middle East          1
East Asia            1
Name: count, dtype: int64

In [3]:
from ml.recommender.hybrid import get_recommender
rec = get_recommender()
def show(**kw):
    return pd.DataFrame(rec.recommend(top_n=8, **kw))[["destination","score","predicted_cost","budget_fit","top_activities"]]
show(interests=["beach","food"], budget=30000, duration_days=5, month=12)

,destination,score,predicted_cost,budget_fit,top_activities
0,"Alibag, India",0.7578,9076,within,"[history, beach, architecture]"
1,"Auroville, India",0.7500,6875,within,"[beach, food, relaxation]"
2,"Alappuzha, India",0.7333,6875,within,"[beach, backwaters, food]"
3,"Ganpatipule, India",0.7163,9076,within,"[beach, trekking, temples]"
4,"Mithapur, India",0.7129,8795,within,"[beach, nature, relaxation]"
5,"Visakhapatnam, India",0.7022,26436,within,"[beach, food, museums]"
6,"Talasari, India",0.7009,9076,within,"[beach, food, temples]"
7,"Covelong, India",0.6987,6875,within,"[beach, shopping, temples]"


In [4]:
show(interests=["history","temples"], budget=40000, duration_days=6, month=1, origin_region="North India")

,destination,score,predicted_cost,budget_fit,top_activities
0,"Anjar, India",0.8098,10554,within,"[temples, relaxation, history]"
1,"Kalol, India",0.7936,10554,within,"[history, temples, nature]"
2,"Darbhanga, India",0.7803,19560,within,"[temples, history, food]"
3,"Datia, India",0.7666,8035,within,"[history, temples, relaxation]"
4,"Junnar, India",0.7631,10891,within,"[history, temples, relaxation]"
5,"Chittorgarh, India",0.7454,20436,within,"[history, temples, food]"
6,"Amreli, India",0.7412,10554,within,"[temples, history, nature]"
7,"Faridabad, India",0.7389,13820,within,"[temples, history, nightlife]"


In [5]:
show(interests=["trekking","mountains","snow"], budget=50000, duration_days=7, month=6, origin_region="South India")

,destination,score,predicted_cost,budget_fit,top_activities
0,"Sabarimala, India",0.8412,9625,within,"[trekking, mountains, nature]"
1,"Auli, India",0.7339,18773,within,"[snow, relaxation, photography]"
2,"Kalpetta, India",0.7313,9238,within,"[mountains, trekking, nature]"
3,"Chopta, India",0.6858,12314,within,"[temples, nature, snow]"
4,"Kausani, India",0.6792,9375,within,"[snow, relaxation, mountains]"
5,"Khopoli, India",0.6742,12707,within,"[temples, snow, mountains]"
6,"Araku Valley, India",0.6555,9625,within,"[nature, mountains, photography]"
7,"Courtallam, India",0.6528,9625,within,"[temples, mountains, nature]"


In [6]:
from ml.db import fetch_df
R = fetch_df("SELECT eml, destination, rating FROM interactions WHERE interaction_type='rating'")
R = R[R.rating>=4]
rng = np.random.default_rng(0); hits=tot=0; K=20
for eml, g in R.groupby("eml"):
    if len(g) < 4: continue
    held = g.sample(1, random_state=int(rng.integers(1e6))).destination.iloc[0]
    liked = [d for d in g.destination if d != held and d in rec.profiles]
    if held not in rec.dest_names or not liked: continue
    acts = pd.Series([a for d in liked for a in rec._top_activities(d, 3)]).value_counts().index[:3].tolist()
    top = [r["destination"] for r in rec.recommend(interests=acts, top_n=K)]
    hits += held in top; tot += 1
print(f"hit-rate@{K} = {hits/tot:.2%} over {tot} users (random baseline ~ {K/len(rec.dest_names):.2%})")
import scrapbook as sb; sb.glue("hit_rate_at_20", hits/tot)

hit-rate@20 = 7.50% over 40 users (random baseline ~ 3.57%)
